# 02 · Silver: limpeza, correções e qualidade

Como parte central da arquitetura medalhão, esta etapa realiza a transição para a tabela silver.chamados_hora, transformando os dados brutos e textuais da bronze em uma base limpa, estruturada com uma linha por chamado e com campos padronizados (como horários convertidos para timestamp).

Seguindo rigorosamente o princípio de que nada é descartado em silêncio, o processo trata os problemas de qualidade previstos no enunciado, como registros sem horário de início ou finalização, aplicando as decisões de integridade técnica do grupo. Cada inconsistência é tratada de forma transparente por meio de correções rastreáveis (mantendo o valor original salvo), flags de qualidade (dq_*) ou o envio para quarentena com o respectivo motivo, garantindo uma base limpa e auditável que servirá de base segura para as métricas da camada Gold.

## O que foi encontrado no `Chamados_Hora.CSV`

| # | Achado | Qtd | Tratamento |
|---|---|---|---|
| 1 | Ano da abertura errado (2027, com início e fim em 2025 ou 2026) | 198 | Corrige o ano, mantém `hora_abertura_original` e marca `corr_ano_abertura = true` |
| 2 | Sem hora de início, mas com finalização | 3.414 (2,6%) | `situacao_atendimento = NAO_ATENDIDO` (provável abandono); fica fora das métricas de espera e tempo médio de atendimento (TMA) |
| 3 | Finalização antes do início | 1 | Mantida como veio e sinalizada (`dq_fim_antes_inicio`); fica fora das métricas de atendimento |
| 4 | Espera acima de 1h / atendimento acima de 2h | 58 / 130 | Apenas sinalizados (`dq_*_outlier`) |
| 5 | Encoding Windows-1252 e datas no formato `dd/MM/yyyy às HH:mm:ss` | todas | Parse com vários formatos; o "às" é removido antes da conversão |

## Como cada chamado é classificado

| `situacao_atendimento` | Regra |
|---|---|
| `ATENDIDO` | Tem hora de início e de finalização |
| `NAO_ATENDIDO` | Sem início, com finalização (provável abandono) |
| `SEM_FINALIZACAO` | Tem início, mas não tem finalização |
| `SEM_HORARIOS` | Não tem início nem finalização |

O `NAO_ATENDIDO` é uma hipótese: dialoga com a observação da Coordenadora de Customer Success de que muitos clientes desistem antes de serem atendidos, e deve ser confirmada cruzando com `Resolvido` na base de chamados.

`dq_status` resume a qualidade do registro, nesta ordem de prioridade: `INCONSISTENTE` (horário ilegível, abertura nula, início antes da abertura, fim antes do início ou fora do período), `INCOMPLETO` (falta início, fim ou cliente), `CORRIGIDO` (houve correção de ano) e `OK`.

## Tabelas geradas

| Tabela | Conteúdo |
|---|---|
| `silver.chamados_hora` | 1 linha por `id_chamado`, tipada, com correções e flags |
| `silver.chamados_hora_quarentena` | Linhas da Bronze que ficaram de fora (ID inválido ou duplicado), com o motivo |

## Idempotência
A Silver é recalculada por inteiro a partir da Bronze e aplicada com `MERGE` por `id_chamado`: só insere IDs novos e só atualiza linhas cujo `_silver_hash` mudou.

## Fonte da verdade
Para o **ano**, valem os horários de início e finalização, validados pela cronologia dos IDs. Para os **tempos**, valem os horários registrados: espera = início − abertura, atendimento = finalização − início. Cada correção tem uma chave liga/desliga nos parâmetros (`CORRIGIR_ANO_ABERTURA`, `CORRIGIR_VIRADA_DIA_FIM`).

## Parâmetros
Janela de dados do case (jan/2025 a jul/2026), limites usados para sinalizar outliers e as chaves liga/desliga das correções. A correção de virada de meia-noite fica desligada por padrão.

In [0]:
import os, re, uuid, codecs
from datetime import datetime
from pyspark.sql import functions as F, Window, DataFrame

CATALOG        = "workspace"
SCHEMA_BRONZE  = "bronze"
SCHEMA_SILVER  = "silver"
SCHEMA_GOLD    = "gold"
SCHEMA_OPS     = "ops"            # log de execução / observabilidade

T_BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}.chamados_hora"
T_SILVER = f"{CATALOG}.{SCHEMA_SILVER}.chamados_hora"
T_QUAR   = f"{CATALOG}.{SCHEMA_SILVER}.chamados_hora_quarentena"
T_LOG    = f"{CATALOG}.{SCHEMA_OPS}.pipeline_run_log"

# Regras de negócio / qualidade (documentadas no README)
PERIODO_INICIO = "2025-01-01 00:00:00"   # janela de dados do case (jan/2025 a jul/2026)
PERIODO_FIM    = "2026-07-31 23:59:59"
CORRIGIR_ANO_ABERTURA   = True    # abertura com ano errado (ex.: 2027) -> usa o ano de início/fim
CORRIGIR_VIRADA_DIA_FIM = False   # finalização "antes" do início por virada de meia-noite -> False = mantém o horário como veio
LIMIAR_DIAS_ANO_DIVERGENTE = 180  # abertura > início/fim em >= 180 dias => erro de ANO (e não relógio desajustado)
MAX_DURACAO_CORRECAO_SEG   = 86400  # após corrigir o ano, o chamado (abertura → fim) deve durar <= 24h
MAX_ATENDIMENTO_SEG        = 7200   # > 2h  => outlier de atendimento (apenas sinalizado, nunca descartado)
MAX_ESPERA_SEG             = 3600   # > 1h  => outlier de espera       (apenas sinalizado, nunca descartado)

# Formatos de data aceitos, em ordem de tentativa. O original é "29/06/2026 às 14:12:48"; o conector ("às") é removido antes do parse, então basta o formato d/M/yyyy H:mm:ss.
FORMATOS_TS = ["d/M/yyyy H:mm:ss", "d/M/yyyy H:mm",
               "yyyy-M-d H:mm:ss", "yyyy-M-d H:mm",
               "yyyy-MM-dd'T'HH:mm:ss", "yyyy-MM-dd'T'HH:mm:ss.SSS"]

# Layout esperado do arquivo (nomes já normalizados: minúsculo, sem acento)
COLS_ESPERADAS = ["id_chamado", "id_cliente", "hora_abertura_chamado",
                  "hora_inicio_atendimento", "hora_finalizacao_atendimento"]

try: 
    spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
except Exception:
    pass

def registrar_log(etapa: str, tabela: str, linhas_lidas=None, linhas_gravadas=None,
                  status: str = "OK", detalhe: str = "", fingerprint=None):
    """Trilha de auditoria de cada execução (append: é log, não dado de negócio)."""
    row = [(RUN_ID, datetime.now(), etapa, tabela, linhas_lidas, linhas_gravadas, status, detalhe, fingerprint)]
    schema = ("run_id string, executado_em timestamp, etapa string, tabela string, linhas_lidas long, "
              "linhas_gravadas long, status string, detalhe string, fingerprint string")
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(T_LOG)


RUN_ID = uuid.uuid4().hex[:12]

## Funções de transformação (Bronze → Silver)
Toda a regra está em `construir_silver`, que recebe a Bronze e devolve a Silver e a quarentena. A função é determinística (mesma entrada, mesma saída) e segue esta ordem:
1. envia para a quarentena as linhas sem `id_chamado` válido;
2. remove duplicados por `id_chamado`, mantendo o registro mais completo;
3. converte tipos (ID, cliente, horários);
4. corrige o ano da abertura;
5. corrige virada de meia-noite na finalização (opcional);
6. calcula tempos e situação do atendimento;
7. cria as flags de qualidade e o `dq_status`.

In [0]:
def _txt(nome: str):
    """trim + vazio vira NULL."""
    t = F.trim(F.col(nome))
    return F.when(t == "", F.lit(None)).otherwise(t)


def parse_ts(nome: str):
    """Converte texto em timestamp tentando vários formatos. Tolera o conector 'às' (e variações de encoding como '?s')."""
    base = F.regexp_replace(_txt(nome), r"^(\d{1,2}/\d{1,2}/\d{4})\D+(\d{1,2}:\d{2}(:\d{2})?)$", "$1 $2")
    return F.coalesce(*[F.try_to_timestamp(base, F.lit(f)) for f in FORMATOS_TS])


def _flag(cond):
    """Coluna booleana nunca nula (NULL conta como False)."""
    return F.coalesce(cond, F.lit(False))


def construir_silver(df_b: DataFrame):
    """Bronze → (silver, quarentena). Função pura e determinística: mesma entrada gera mesma saída."""
    d = (df_b
         .withColumn("_id_txt", F.regexp_replace(_txt("id_chamado"), r"\.0+$", ""))
         .withColumn("_id_ok", _flag(F.col("_id_txt").rlike(r"^\d{1,18}$"))))

    # 4.1 Quarentena: ID ausente/ inválido
    invalidos = d.where(~F.col("_id_ok")).withColumn("motivo", F.lit("ID_CHAMADO_AUSENTE_OU_INVALIDO"))

    # 4.2 Deduplicação por ID_Chamado: fica o registro mais completo; empate → ingestão mais recente → chave
    validos = d.where(F.col("_id_ok"))
    n_nulos = sum([F.when(_txt(c).isNull(), 1).otherwise(0) for c in COLS_ESPERADAS[1:]])
    w = Window.partitionBy("_id_txt").orderBy(n_nulos.asc(), F.col("_ingested_at").desc(), F.col("_bronze_key").asc())
    w_cnt = Window.partitionBy("_id_txt")
    validos = validos.withColumn("_rn", F.row_number().over(w)).withColumn("_qtd_bronze", F.count("*").over(w_cnt))
    duplicados = validos.where(F.col("_rn") > 1).withColumn("motivo", F.lit("ID_CHAMADO_DUPLICADO_DESCARTADO"))

    quarentena = (invalidos.unionByName(duplicados.drop("_rn", "_qtd_bronze"))
                  .select("_bronze_key", "motivo", "id_chamado", "id_cliente", "hora_abertura_chamado",
                          "hora_inicio_atendimento", "hora_finalizacao_atendimento",
                          "_source_file", "_ingested_at")
                  .withColumn("_processed_at", F.current_timestamp()))

    s = validos.where(F.col("_rn") == 1)

    # 4.3 Tipagem e normalização
    cliente = F.regexp_replace(F.regexp_replace(_txt("id_cliente"), r"\.0+$", ""), r"[\.\-\s/]", "")
    s = (s.withColumn("id_chamado", F.col("_id_txt").cast("bigint"))
          .withColumn("id_cliente", F.when(cliente.rlike(r"^\d+$"), cliente))
          .withColumn("_raw_ab", _txt("hora_abertura_chamado").isNotNull())
          .withColumn("_raw_in", _txt("hora_inicio_atendimento").isNotNull())
          .withColumn("_raw_fi", _txt("hora_finalizacao_atendimento").isNotNull())
          .withColumn("_ab", parse_ts("hora_abertura_chamado"))
          .withColumn("_in", parse_ts("hora_inicio_atendimento"))
          .withColumn("_fi", parse_ts("hora_finalizacao_atendimento")))

    # 4.4 Correção do ANO da abertura
    s = (s.withColumn("_ref", F.coalesce("_in", "_fi"))
          .withColumn("_fim_ref", F.coalesce("_fi", "_in"))
          .withColumn("_k", F.year("_ref") - F.year("_ab"))
          .withColumn("_a1", F.expr("_ab + make_interval(_k, 0, 0, 0, 0, 0, 0)"))
          .withColumn("_a2", F.when(F.col("_a1") > F.col("_ref"),
                                    F.expr("_ab + make_interval(_k - 1, 0, 0, 0, 0, 0, 0)")).otherwise(F.col("_a1")))
          .withColumn("corr_ano_abertura",
                      F.lit(CORRIGIR_ANO_ABERTURA) & _flag((F.datediff("_ab", "_ref") >= LIMIAR_DIAS_ANO_DIVERGENTE) &
                            ((F.col("_fim_ref").cast("long") - F.col("_a2").cast("long")).between(0, MAX_DURACAO_CORRECAO_SEG))))
          .withColumn("_ab_c", F.when(F.col("corr_ano_abertura"), F.col("_a2")).otherwise(F.col("_ab"))))

    # 4.5 Correção de virada de meia-noite na finalização
    s = (s.withColumn("_ref2", F.coalesce("_in", "_ab_c"))
          .withColumn("_fi_mais1d", F.col("_fi") + F.expr("INTERVAL 1 DAY"))
          .withColumn("corr_virada_dia_fim",
                      F.lit(CORRIGIR_VIRADA_DIA_FIM) & _flag((F.col("_fi") < F.col("_ref2")) & (F.col("_fi_mais1d") >= F.col("_ref2")) &
                            ((F.col("_fi_mais1d").cast("long") - F.col("_ref2").cast("long")) <= MAX_ATENDIMENTO_SEG)))
          .withColumn("_fi_c", F.when(F.col("corr_virada_dia_fim"), F.col("_fi_mais1d")).otherwise(F.col("_fi"))))

    # 4.6 Campos derivados (calculados dos horários tratados)
    s = (s.withColumnRenamed("_ab", "hora_abertura_original")
          .withColumnRenamed("_fi", "hora_finalizacao_original")
          .withColumn("hora_abertura_chamado", F.col("_ab_c"))
          .withColumn("hora_inicio_atendimento", F.col("_in"))
          .withColumn("hora_finalizacao_atendimento", F.col("_fi_c"))
          .withColumn("espera_seg", F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
          .withColumn("atendimento_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
          .withColumn("duracao_total_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
          .withColumn("situacao_atendimento",
                      F.when(F.col("hora_inicio_atendimento").isNotNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "ATENDIDO")
                       .when(F.col("hora_inicio_atendimento").isNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "NAO_ATENDIDO")
                       .when(F.col("hora_inicio_atendimento").isNotNull(), "SEM_FINALIZACAO")
                       .otherwise("SEM_HORARIOS"))
          .withColumn("tempo_ate_desistencia_seg",
                      F.when(F.col("situacao_atendimento") == "NAO_ATENDIDO", F.col("duracao_total_seg")))
          .withColumn("espera_valida", _flag(F.col("espera_seg") >= 0))
          .withColumn("atendimento_valido", _flag(F.col("atendimento_seg") >= 0))
          .withColumn("data_abertura", F.to_date("hora_abertura_chamado"))
          .withColumn("hora_abertura", F.hour("hora_abertura_chamado"))
          .withColumn("ano_mes", F.date_format("hora_abertura_chamado", "yyyy-MM")))

    # 4.7 Flags de qualidade
    s = (s.withColumn("dq_id_duplicado", F.col("_qtd_bronze") > 1)
          .withColumn("dq_id_cliente_nulo", F.col("id_cliente").isNull())
          .withColumn("dq_abertura_nula", F.col("hora_abertura_chamado").isNull())
          .withColumn("dq_inicio_nulo", F.col("hora_inicio_atendimento").isNull())
          .withColumn("dq_fim_nulo", F.col("hora_finalizacao_atendimento").isNull())
          .withColumn("dq_ts_ilegivel",
                      (F.col("_raw_ab") & F.col("hora_abertura_original").isNull()) |
                      (F.col("_raw_in") & F.col("hora_inicio_atendimento").isNull()) |
                      (F.col("_raw_fi") & F.col("hora_finalizacao_original").isNull()))
          .withColumn("dq_inicio_antes_abertura", _flag(F.col("espera_seg") < 0))
          .withColumn("dq_fim_antes_inicio", _flag(F.col("atendimento_seg") < 0))
          .withColumn("dq_fora_periodo", _flag((F.col("hora_abertura_chamado") < F.lit(PERIODO_INICIO).cast("timestamp")) |
                                               (F.col("hora_abertura_chamado") > F.lit(PERIODO_FIM).cast("timestamp"))))
          .withColumn("dq_espera_outlier", _flag(F.col("espera_seg") > MAX_ESPERA_SEG))
          .withColumn("dq_atendimento_outlier", _flag(F.col("atendimento_seg") > MAX_ATENDIMENTO_SEG)))

    s = s.withColumn("dq_status",
                     F.when(F.col("dq_ts_ilegivel") | F.col("dq_abertura_nula") | F.col("dq_inicio_antes_abertura") |
                            F.col("dq_fim_antes_inicio") | F.col("dq_fora_periodo"), "INCONSISTENTE")
                      .when(F.col("dq_inicio_nulo") | F.col("dq_fim_nulo") | F.col("dq_id_cliente_nulo"), "INCOMPLETO")
                      .when(F.col("corr_ano_abertura") | F.col("corr_virada_dia_fim"), "CORRIGIDO")
                      .otherwise("OK"))

    cols_neg = ["id_chamado", "id_cliente", "hora_abertura_chamado", "hora_inicio_atendimento",
                "hora_finalizacao_atendimento", "hora_abertura_original", "hora_finalizacao_original",
                "espera_seg", "atendimento_seg", "duracao_total_seg", "tempo_ate_desistencia_seg",
                "situacao_atendimento", "espera_valida", "atendimento_valido",
                "data_abertura", "hora_abertura", "ano_mes",
                "corr_ano_abertura", "corr_virada_dia_fim",
                "dq_id_duplicado", "dq_id_cliente_nulo", "dq_abertura_nula", "dq_inicio_nulo", "dq_fim_nulo",
                "dq_ts_ilegivel", "dq_inicio_antes_abertura", "dq_fim_antes_inicio", "dq_fora_periodo",
                "dq_espera_outlier", "dq_atendimento_outlier", "dq_status"]
    # hash só das colunas de negócio: detecta mudança real e evita regravar linhas iguais (idempotência)
    hash_neg = F.sha2(F.concat_ws("\u001f", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000")) for c in cols_neg]), 256)
    silver = (s.select(*cols_neg, "_bronze_key", "_source_file", "_ingested_at")
                .withColumn("_processed_at", F.current_timestamp())
                .withColumn("_silver_hash", hash_neg))
    return silver, quarentena

## 1. Construir Silver e quarentena a partir da Bronze
Aplica a função sobre a Bronze inteira e deixa os dois resultados prontos para a gravação.

In [0]:
df_b = spark.table(T_BRONZE)
linhas_bronze = df_b.count()
silver, quarentena = construir_silver(df_b)

silver.createOrReplaceTempView("v_silver_novo")
quarentena.createOrReplaceTempView("v_quar_novo")
print(f"Bronze: {linhas_bronze:,} linhas")

Bronze: 131,838 linhas


## 2. Gravação da Silver (MERGE idempotente)
Insere os chamados novos e atualiza apenas os que mudaram. Ao final, imprime quantas linhas foram inseridas e atualizadas e registra no log.

In [0]:
if not spark.catalog.tableExists(T_SILVER):
    silver.limit(0).write.format("delta").saveAsTable(T_SILVER)

spark.sql(f"""
MERGE INTO {T_SILVER} AS t
USING v_silver_novo AS s
ON t.id_chamado = s.id_chamado
WHEN MATCHED AND t._silver_hash <> s._silver_hash THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

m = spark.sql(f"DESCRIBE HISTORY {T_SILVER} LIMIT 1").collect()[0]["operationMetrics"]
ins, upd = int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))
n_silver = spark.table(T_SILVER).count()
spark.sql(f"COMMENT ON TABLE {T_SILVER} IS 'Silver: 1 linha por chamado, tipada, com correcoes rastreaveis e flags de qualidade (dq_*).'")
print(f"Silver -> inseridas: {ins:,} | atualizadas: {upd:,} | total: {n_silver:,}")
registrar_log("silver", T_SILVER, linhas_bronze, ins + upd, "OK", f"inseridas={ins}; atualizadas={upd}; total={n_silver}")

Silver -> inseridas: 0 | atualizadas: 0 | total: 131,838


## 3. Quarentena
Guarda as linhas que não puderam entrar na Silver, com o motivo (`ID_CHAMADO_AUSENTE_OU_INVALIDO` ou `ID_CHAMADO_DUPLICADO_DESCARTADO`). Se houver linhas na quarentena, o log registra status `ALERTA`, o que é esperado quando o arquivo tem IDs duplicados.

In [0]:
if not spark.catalog.tableExists(T_QUAR):
    quarentena.limit(0).write.format("delta").saveAsTable(T_QUAR)

spark.sql(f"""
MERGE INTO {T_QUAR} AS t
USING v_quar_novo AS s
ON t._bronze_key = s._bronze_key
WHEN NOT MATCHED THEN INSERT *
""")
n_quar = spark.table(T_QUAR).count()
spark.sql(f"COMMENT ON TABLE {T_QUAR} IS 'Quarentena: linhas da Bronze descartadas da Silver (ID invalido ou duplicado), com motivo.'")
print(f"Quarentena: {n_quar:,} linhas")
registrar_log("silver_quarentena", T_QUAR, linhas_bronze, n_quar, "OK" if n_quar == 0 else "ALERTA", f"total={n_quar}")

Quarentena: 0 linhas


## 4. Conferência
Resumo da qualidade (quantidade de chamados por `dq_status` e por situação do atendimento) e exemplos de chamados corrigidos, comparando o horário original com o tratado.

In [0]:
display(spark.table(T_SILVER).groupBy("dq_status", "situacao_atendimento").count().orderBy("dq_status"))

dq_status,situacao_atendimento,count
CORRIGIDO,ATENDIDO,192
INCOMPLETO,NAO_ATENDIDO,3414
INCONSISTENTE,ATENDIDO,1
OK,ATENDIDO,128231


In [0]:
# Exemplos das correções (original × tratado)
display(spark.table(T_SILVER)
        .where("corr_ano_abertura OR corr_virada_dia_fim")
        .select("id_chamado", "hora_abertura_original", "hora_abertura_chamado",
                "hora_finalizacao_original", "hora_finalizacao_atendimento", "corr_ano_abertura", "corr_virada_dia_fim")
        .limit(20))

id_chamado,hora_abertura_original,hora_abertura_chamado,hora_finalizacao_original,hora_finalizacao_atendimento,corr_ano_abertura,corr_virada_dia_fim
106858,2027-05-20T18:40:51.000-03:00,2026-05-20T18:40:51.000-03:00,2026-05-20T18:42:15.000-03:00,2026-05-20T18:42:15.000-03:00,true,false
114561,2027-06-02T08:40:53.000-03:00,2026-06-02T08:40:53.000-03:00,2026-06-02T09:10:53.000-03:00,2026-06-02T09:10:53.000-03:00,true,false
11663,2027-03-24T11:33:01.000-03:00,2025-03-24T11:33:01.000-03:00,2025-03-24T12:00:39.000-03:00,2025-03-24T12:00:39.000-03:00,true,false
117807,2027-06-25T12:52:06.000-03:00,2026-06-25T12:52:06.000-03:00,2026-06-25T12:54:05.000-03:00,2026-06-25T12:54:05.000-03:00,true,false
118939,2027-06-17T09:15:18.000-03:00,2026-06-17T09:15:18.000-03:00,2026-06-17T09:17:35.000-03:00,2026-06-17T09:17:35.000-03:00,true,false
16066,2027-04-23T19:10:34.000-03:00,2025-04-23T19:10:34.000-03:00,2025-04-23T19:12:04.000-03:00,2025-04-23T19:12:04.000-03:00,true,false
16842,2027-04-29T20:05:27.000-03:00,2025-04-29T20:05:27.000-03:00,2025-04-29T20:11:03.000-03:00,2025-04-29T20:11:03.000-03:00,true,false
18701,2027-04-30T23:02:53.000-03:00,2025-04-30T23:02:53.000-03:00,2025-04-30T23:05:10.000-03:00,2025-04-30T23:05:10.000-03:00,true,false
27689,2027-06-26T19:12:24.000-03:00,2025-06-26T19:12:24.000-03:00,2025-06-26T19:33:59.000-03:00,2025-06-26T19:33:59.000-03:00,true,false
33754,2027-07-07T10:21:19.000-03:00,2025-07-07T10:21:19.000-03:00,2025-07-07T10:30:53.000-03:00,2025-07-07T10:30:53.000-03:00,true,false
